In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score,recall_score, precision_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
# Load & combine labeled data
import glob
import os

files = glob.glob("data/raw/*_labeled.csv")
dfs = [pd.read_csv(f, index_col="timestamp", parse_dates=True) for f in files]
combined = pd.concat(dfs).dropna()

# Add extra features
combined["high_low_range"]  = (combined["high"] - combined["low"]) / combined["low"]
combined["close_open_diff"] = (combined["close"] - combined["open"]) / combined["open"]

# ✅ Correct features aligned with app.py
features = ["open", "high", "low", "close", "volume", "trades",
            "high_low_range", "close_open_diff"]

X = combined[features]
y = combined["is_pump_dump"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Total rows:", len(combined))
print(combined["is_pump_dump"].value_counts())

Total rows: 4965
is_pump_dump
0    4702
1     263
Name: count, dtype: int64


In [3]:
# (Removed redundant feature redefinition)

In [4]:
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)
X_resampled, y_resampled = sm.fit_resample(X_train, y_train)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE : {pd.Series(y_resampled).value_counts().to_dict()}")


Before SMOTE: {0: 3762, 1: 210}
After SMOTE : {0: 3762, 1: 3762}


In [5]:
model = XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss")
model.fit(X_resampled, y_resampled)

# Save model
os.makedirs("models", exist_ok=True)
model.save_model("models/xgb_model.json")
print("✅ Model saved to models/xgb_model.json")

In [6]:
y_pred_xg = model.predict(X_test)

print("Accuracy score:",accuracy_score(y_test, y_pred_xg))
print("Precision score:",precision_score(y_test, y_pred_xg))
print("Recall score:",recall_score(y_test, y_pred_xg))
print("F1 score:",f1_score(y_test, y_pred_xg))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xg))

Accuracy score: 0.998992950654582
Precision score: 0.9814814814814815
Recall score: 1.0
F1 score: 0.9906542056074766
Confusion Matrix:
[[939   1]
 [  0  53]]


In [7]:
# Add this to check overfitting
train_pred = model.predict(X_resampled)
print("Train Recall:", recall_score(y_resampled, train_pred))
print("Test Recall :", recall_score(y_test, y_pred_xg))

Train Recall: 1.0
Test Recall : 1.0


In [8]:

rf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)

In [9]:
rf.fit(X_resampled, y_resampled)
y_pred_rf = rf.predict(X_test)
print("Random Forest Accuracy score:",accuracy_score(y_test, y_pred_rf))
print("Random Forest Precision score:",precision_score(y_test, y_pred_rf))
print("Random Forest Recall score:",recall_score(y_test, y_pred_rf))
print("Random Forest F1 score:",f1_score(y_test, y_pred_rf))
print("Random Forest Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))


Random Forest Accuracy score: 1.0
Random Forest Precision score: 1.0
Random Forest Recall score: 1.0
Random Forest F1 score: 1.0
Random Forest Confusion Matrix:
[[940   0]
 [  0  53]]


In [10]:
train_pred = rf.predict(X_resampled)
print("Train Recall:", recall_score(y_resampled, train_pred))
print("Test Recall :", recall_score(y_test, y_pred_rf))

Train Recall: 1.0
Test Recall : 1.0


In [11]:

lgbm = LGBMClassifier(class_weight="balanced", random_state=42)

In [12]:
lgbm.fit(X_resampled, y_resampled)
y_pred_lgbm = lgbm.predict(X_test)
print("LightGBM Accuracy score:",accuracy_score(y_test, y_pred_lgbm))
print("LightGBM Precision score:",precision_score(y_test, y_pred_lgbm))
print("LightGBM Recall score:",recall_score(y_test, y_pred_lgbm))
print("LightGBM F1 score:",f1_score(y_test, y_pred_lgbm))
print("LightGBM Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lgbm))

[LightGBM] [Info] Number of positive: 3762, number of negative: 3762
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000176 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2040
[LightGBM] [Info] Number of data points in the train set: 7524, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

In [13]:
train_pred = lgbm.predict(X_resampled)
print("Train Recall:", recall_score(y_resampled, train_pred))
print("Test Recall :", recall_score(y_test, y_pred_lgbm))

Train Recall: 1.0
Test Recall : 1.0


In [14]:
from sklearn.svm import SVC
svm = SVC(class_weight="balanced", probability=True, random_state=42)

In [15]:
svm.fit(X_resampled, y_resampled)
y_pred_svm = svm.predict(X_test)
print("SVM Accuracy score:",accuracy_score(y_test, y_pred_svm))
print("SVM Precision score:",precision_score(y_test, y_pred_svm))
print("SVM Recall score:",recall_score(y_test, y_pred_svm))
print("SVM F1 score:",f1_score(y_test, y_pred_svm))
print("SVM Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

SVM Accuracy score: 0.9113796576032226
SVM Precision score: 0.18181818181818182
SVM Recall score: 0.18867924528301888
SVM F1 score: 0.18518518518518517
SVM Confusion Matrix:
[[895  45]
 [ 43  10]]


In [16]:
train_pred = svm.predict(X_resampled)
print("Train Recall:", recall_score(y_resampled, train_pred))
print("Test Recall :", recall_score(y_test, y_pred_svm))

Train Recall: 0.1796916533758639
Test Recall : 0.18867924528301888
